# Notebook 17 — MELU-Δt vs Published Baselines
## Clean version — two protocol fixes applied

### What changed vs previous version

| Issue | Previous | This version |
|---|---|---|
| MCD fit on | ALL inliers (incl. test) | Training inliers ONLY |
| tau definition | `mean(dm)` ≈ 50th pct → gate=52% | 75th percentile → gate≈25% |

### Background

All four reference papers use the Zong et al. (ICLR 2018) benchmark protocol
on the same four canonical tabular datasets. Their published F1 scores are
cited directly — re-implementing each method's custom architecture and tuning
pipeline is not feasible and not standard practice.

### Datasets

| Dataset | Dim | N | Anomaly% |
|---|---|---|---|
| Arrhythmia | 274 | 452 | 14.6% |
| Thyroid | 6 | 7200 | 2.5% |
| KDD Cup 99 | ~118 | ~494k | 20% |
| KDDRev | ~118 | ~494k | 20% |

### Getting real .mat files (required for valid comparison)

```bash
pip install pyod        # auto-downloads arrhythmia.mat + thyroid.mat
```
Or place them manually in a `data/` folder:
- `data/arrhythmia.mat` → http://odds.cs.stonybrook.edu/arrhythmia-dataset/
- `data/thyroid.mat`    → http://odds.cs.stonybrook.edu/thyroid-disease-dataset/

KDD loads from sklearn automatically (no download needed).

Without .mat files Cell 2 uses faithful simulations but marks results `(SIM)` —
those must **not** be compared against published numbers.

### Published baseline F1

| Dataset | DAGMM [1] | GOAD [2] | NeuTraL-AD [3] | ICL [4] |
|---|---|---|---|---|
| Arrhythmia | 49.8 | 62.8 | 51.0 | **67.2** |
| Thyroid | **93.7** | 85.3 | 76.1 | 93.8 |
| KDD | 93.7 | 99.0 | 99.4 | **99.9** |
| KDDRev | 86.6 | 97.8 | 98.5 | **98.9** |

[1] Zong et al. ICLR 2018  [2] Bergman & Hoshen ICLR 2020  
[3] Qiu et al. ICML 2021    [4] Shenkar & Wolf ICLR 2022


## Cell 1 — Imports and configuration

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import betainc, gammaln
from scipy.io import loadmat, savemat
from scipy.stats import chi2 as chi2_dist
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import warnings, time, os, urllib.request
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

# ── Protocol (Zong et al. ICLR 2018) ─────────────────────────────────────────
N_RUNS_SMALL = 20     # Arrhythmia, Thyroid
N_RUNS_LARGE = 10     # KDD, KDDRev
TRAIN_FRAC   = 0.50   # 50% of normal for training, NO anomalies
LR           = 1e-3
EPOCHS_PRE   = 60     # Stage 1: ELU pretraining
EPOCHS_FINE  = 80     # Stage 3: MELU fine-tuning
LAM_BCE      = 0.5    # BCE pseudo-label loss weight
PCT_PSEUDO   = 85     # percentile for pseudo-label threshold

# ── Two fixes vs earlier notebooks ───────────────────────────────────────────
# FIX 1: MCD on TRAINING inliers only (not all inliers)
#         → avoids test-inlier information in gate signal
# FIX 2: tau = chi2 quantile at 75% (not mean)
#         → gate fires on top 25% of inlier distribution, not top 50%
TAU_PERCENTILE = 75   # gate fires on inliers with dm > 75th percentile

def lat_for(dim): return max(4, min(dim // 2, 16))
def hid_for(dim): return max(64, dim * 4)

print(f"PyTorch {torch.__version__} | Numpy {np.__version__}")
print()
print("Protocol: Zong et al. ICLR 2018")
print(f"  Train: {TRAIN_FRAC:.0%} normal only | Test: {1-TRAIN_FRAC:.0%} normal + ALL anomalies")
print(f"  Runs: {N_RUNS_SMALL} (small) / {N_RUNS_LARGE} (large)")
print(f"  Metric: F1 (threshold = true anomaly count) + AUROC")
print()
print("Two fixes applied vs previous version:")
print("  FIX 1: MCD fit on TRAINING inliers only (no test leakage)")
print(f"  FIX 2: tau = {TAU_PERCENTILE}th percentile of MCD distances (not mean)")
print(f"         → gate fires on top {100-TAU_PERCENTILE}% of inliers (target: 20-30%)")


PyTorch 2.5.1+cu121 | Numpy 1.26.4

Protocol: Zong et al. ICLR 2018
  Train: 50% normal only | Test: 50% normal + ALL anomalies
  Runs: 20 (small) / 10 (large)
  Metric: F1 (threshold = true anomaly count) + AUROC

Two fixes applied vs previous version:
  FIX 1: MCD fit on TRAINING inliers only (no test leakage)
  FIX 2: tau = 75th percentile of MCD distances (not mean)
         → gate fires on top 25% of inliers (target: 20-30%)


## Cell 2 — Dataset loading

Downloads automatically via **PyOD** → ODDS URL → faithful simulation (fallback).
Run once; results cached in `data/`.


In [3]:
os.makedirs("data", exist_ok=True)

# ── Download helpers ──────────────────────────────────────────────────────────
ODDS_URLS = {
    "arrhythmia": "http://odds.cs.stonybrook.edu/static/media/releases/datasets/arrhythmia.mat",
    "thyroid":    "http://odds.cs.stonybrook.edu/static/media/releases/datasets/thyroid.mat",
}

def try_pyod(name, path):
    try:
        from pyod.utils.data import load_mat_file
        print(f"  [PyOD] Downloading {name}.mat ...")
        X, y = load_mat_file(name)
        savemat(path, {"X": X, "y": y.reshape(-1, 1)})
        print(f"  [PyOD] Saved {path}  shape={X.shape}")
        return True
    except Exception as e:
        print(f"  [PyOD] Not available: {e}")
        return False

def try_url(name, path):
    url = ODDS_URLS.get(name)
    if not url: return False
    try:
        print(f"  [URL]  Downloading {name}.mat from ODDS ...")
        urllib.request.urlretrieve(url, path)
        print(f"  [URL]  Saved {path}")
        return True
    except Exception as e:
        print(f"  [URL]  Failed: {e}")
        return False

def simulate(name, path):
    stats = {
        "arrhythmia": (386,  66, 274, 0.15),
        "thyroid":    (7018, 182,  6, 0.50),
    }
    n_in, n_out, dim, rho = stats[name]
    print(f"  [SIM]  Simulating {name}  (n_in={n_in} n_out={n_out} dim={dim})")
    print("  *** WARNING: simulation — NOT comparable to published numbers ***")
    np.random.seed(42)
    cov = np.array([[rho**abs(i-j) for j in range(dim)]
                    for i in range(dim)]).astype(np.float32)
    cov += np.eye(dim, dtype=np.float32) * 0.05
    L = np.linalg.cholesky(cov)
    Xi = (np.random.randn(n_in, dim) @ L.T).astype(np.float32)
    shift = np.random.randn(1, dim).astype(np.float32) * 3
    Xo = np.vstack([
        (np.random.randn(n_out//2,      dim) @ L.T + shift).astype(np.float32),
        (np.random.randn(n_out-n_out//2, dim) @ L.T * 2.5 ).astype(np.float32),
    ])
    X = np.vstack([Xi, Xo])
    y = np.array([0]*n_in + [1]*n_out, dtype=np.float32).reshape(-1, 1)
    savemat(path, {"X": X, "y": y})

def load_mat(path):
    mat = loadmat(path)
    X = mat["X"].astype(np.float32)
    y = mat["y"].astype(int).ravel()
    return X[y == 0], X[y == 1]

def get_odds(name):
    path = f"data/{name}.mat"
    if os.path.exists(path):
        print(f"  Found  data/{name}.mat")
        return load_mat(path), "real"
    if try_pyod(name, path) or try_url(name, path):
        return load_mat(path), "real"
    simulate(name, path)
    return load_mat(path), "simulated"

def get_kdd(rev=False, subset=50000):
    from sklearn.datasets import fetch_kddcup99
    print(f"  Loading KDD Cup 99 via sklearn (rev={rev}) ...")
    data  = fetch_kddcup99(subset="SA", percent10=True)
    X_raw = data.data
    y_raw = (data.target == b"normal.").astype(int)
    cat   = [1, 2, 3]
    num   = [i for i in range(X_raw.shape[1]) if i not in cat]
    enc   = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    X_cat = enc.fit_transform(X_raw[:, cat])
    X_num = X_raw[:, num].astype(np.float32)
    X     = np.hstack([X_num, X_cat]).astype(np.float32)
    Xi = X[y_raw == (0 if rev else 1)]
    Xo = X[y_raw == (1 if rev else 0)]
    rng = np.random.RandomState(42)
    ni  = min(len(Xi), subset // 2)
    no  = min(len(Xo), subset // 2)
    return (Xi[rng.choice(len(Xi), ni, replace=False)],
            Xo[rng.choice(len(Xo), no, replace=False)])

# ── Load all four datasets ────────────────────────────────────────────────────
print("=" * 55)
print("Dataset loading")
print("=" * 55)

print("\nArrhythmia:")
(Xi_arr, Xo_arr), arr_src = get_odds("arrhythmia")
print(f"  n_in={len(Xi_arr)}  n_out={len(Xo_arr)}  dim={Xi_arr.shape[1]}  source={arr_src}")

print("\nThyroid:")
(Xi_thy, Xo_thy), thy_src = get_odds("thyroid")
print(f"  n_in={len(Xi_thy)}  n_out={len(Xo_thy)}  dim={Xi_thy.shape[1]}  source={thy_src}")

kdd_ok = kddrev_ok = False
print("\nKDD:")
try:
    Xi_kdd, Xo_kdd = get_kdd(rev=False)
    print(f"  n_in={len(Xi_kdd)}  n_out={len(Xo_kdd)}  dim={Xi_kdd.shape[1]}  source=sklearn")
    kdd_ok = True
except Exception as e:
    print(f"  FAILED: {e}")

print("\nKDDRev:")
try:
    Xi_rev, Xo_rev = get_kdd(rev=True)
    print(f"  n_in={len(Xi_rev)}  n_out={len(Xo_rev)}  dim={Xi_rev.shape[1]}  source=sklearn")
    kddrev_ok = True
except Exception as e:
    print(f"  FAILED: {e}")

print()
if arr_src == "simulated":
    print("*** Arrhythmia: simulated data. pip install pyod for real .mat file.")
if thy_src == "simulated":
    print("*** Thyroid: simulated data. pip install pyod for real .mat file.")


Dataset loading

Arrhythmia:
  Found  data/arrhythmia.mat
  n_in=386  n_out=66  dim=274  source=real

Thyroid:
  Found  data/thyroid.mat
  n_in=7018  n_out=182  dim=6  source=real

KDD:
  Loading KDD Cup 99 via sklearn (rev=False) ...
  n_in=25000  n_out=3377  dim=99  source=sklearn

KDDRev:
  Loading KDD Cup 99 via sklearn (rev=True) ...
  n_in=3377  n_out=25000  dim=99  source=sklearn



## Cell 3 — MELU-Δt model

In [4]:
# ── Student-t CDF (exact PDF gradient, nu=5 fixed) ──────────────────────────
class StudentTCDF(torch.autograd.Function):
    NU = 5.0

    @staticmethod
    def forward(ctx, x):
        nu = StudentTCDF.NU
        xn = x.detach().cpu().numpy()
        z  = nu / (nu + np.clip(xn**2, 1e-30, None))
        ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
        cdf = np.where(xn >= 0, 1. - ib/2., ib/2.)
        ctx.save_for_backward(x)
        return torch.tensor(cdf, dtype=x.dtype, device=x.device)

    @staticmethod
    def backward(ctx, g):
        x, = ctx.saved_tensors
        nu  = StudentTCDF.NU
        xn  = x.detach().cpu().numpy()
        lc  = gammaln((nu+1)/2) - gammaln(nu/2) - .5*np.log(nu*np.pi)
        pdf = np.exp(lc - (nu+1)/2 * np.log(1 + xn**2/nu))
        return g * torch.tensor(pdf, dtype=x.dtype, device=x.device)

tcdf = StudentTCDF.apply


# ── Stage-1 ELU autoencoder ───────────────────────────────────────────────────
class BaseAE(nn.Module):
    def __init__(self, dim, hid, lat):
        super().__init__()
        self.W1  = nn.Linear(dim, hid)
        self.W2  = nn.Linear(hid, lat)
        self.dec = nn.Linear(lat, dim)
        for m in [self.W1, self.W2, self.dec]:
            nn.init.kaiming_normal_(m.weight)
            nn.init.zeros_(m.bias)

    def encode_elu(self, x):
        return self.W2(F.elu(F.silu(self.W1(x))))

    def forward(self, x):
        return self.dec(self.encode_elu(x))

    def recon_err(self, x):
        with torch.no_grad():
            return (x - self(x)).abs().mean(1)


# ── MELU gate (holds frozen MCD parameters) ───────────────────────────────────
class MELUGate(nn.Module):
    def __init__(self, lat):
        super().__init__()
        self.register_buffer("mu",  torch.zeros(lat))
        self.register_buffer("Li",  torch.eye(lat))
        self.register_buffer("tau", torch.tensor(1.5))

    def set_mcd(self, mu_np, Li_np, tau_val):
        dev = self.mu.device
        self.mu  = torch.tensor(mu_np,       dtype=torch.float32, device=dev)
        self.Li  = torch.tensor(Li_np,       dtype=torch.float32, device=dev)
        self.tau = torch.tensor(float(tau_val), device=dev)

    def mahal(self, Z):
        w = (Z - self.mu.unsqueeze(0)) @ self.Li.T
        return w.norm(dim=1)   # Mahalanobis distance


# ── MELU-Δt autoencoder ───────────────────────────────────────────────────────
class MELU_AE(nn.Module):
    # f(h)_i = h_i * T_nu(h_i)                         (m < tau)
    # f(h)_i = h_i * T_nu(h_i) + alpha*sign(h_i)*tanh(beta*(m-tau)) (m >= tau)
    # alpha, beta learnable; gate from frozen Stage-1 Mahalanobis
    def __init__(self, dim, hid, lat):
        super().__init__()
        self.W1      = nn.Linear(dim, hid)
        self.W2      = nn.Linear(hid, lat)
        self.W3      = nn.Linear(lat, dim)
        self.log_alpha = nn.Parameter(torch.log(torch.tensor(1.0)))
        self.log_beta  = nn.Parameter(torch.log(torch.tensor(0.5)))
        self.gate    = MELUGate(lat)
        self.gate_on = False   # disabled during warmup
        for m in [self.W1, self.W2, self.W3]:
            nn.init.kaiming_normal_(m.weight)
            nn.init.zeros_(m.bias)

    @property
    def alpha(self): return self.log_alpha.exp()
    @property
    def beta(self):  return self.log_beta.exp()

    def encode(self, x, enc_frozen=None):
        h1 = F.silu(self.W1(x))
        T1 = h1 * tcdf(h1)                     # Student-t Swish base
        if self.gate_on and enc_frozen is not None:
            with torch.no_grad():
                Zf = enc_frozen.encode_elu(x)  # frozen Stage-1 latent
            m    = self.gate.mahal(Zf)          # Mahalanobis distance
            g    = (m >= self.gate.tau).float().unsqueeze(1)
            amp  = (self.alpha
                    * h1.sign()
                    * torch.tanh(self.beta
                                 * (m.unsqueeze(1) - self.gate.tau)
                                 .clamp(-8, 8)))
            return self.W2(T1 + g * amp)
        return self.W2(T1)

    def forward(self, x, ef=None):
        return self.W3(self.encode(x, ef))

    def recon_err(self, x, ef=None):
        with torch.no_grad():
            return (x - self(x, ef)).abs().mean(1)


# ── Robust MCD estimator ──────────────────────────────────────────────────────
def fast_mcd(Z, hf=0.75, n_starts=6, n_iter=5):
    # MCD estimator. Returns (mu, cov, Li). Reliable when n/d >= 5.
    n, d = Z.shape
    h    = max(int(n * hf), d + 1)
    best_det = np.inf; bm = bc = None

    for _ in range(n_starts):
        idx = np.random.choice(n, h, replace=False)
        sub = Z[idx]
        for _ in range(n_iter):
            mu  = sub.mean(0); dv = sub - mu
            cov = dv.T @ dv / max(len(sub)-1, 1) + 1e-4*np.eye(d)
            Si  = np.linalg.inv(cov)
            ds  = np.sqrt(np.maximum(
                    np.einsum("bi,ij,bj->b", Z-mu, Si, Z-mu), 0))
            idx = np.argsort(ds)[:h]; sub = Z[idx]
        mu  = sub.mean(0); dv = sub - mu
        cov = dv.T @ dv / max(len(sub)-1, 1)
        det = np.linalg.det(cov + 1e-4*np.eye(d))
        if det < best_det:
            best_det = det; bm = mu; bc = cov

    try:
        L  = np.linalg.cholesky(bc + 1e-4*np.eye(d))
        Li = np.linalg.inv(L)
        if np.isnan(Li).any() or np.linalg.cond(Li) > 1e7:
            Li = np.eye(d)
    except Exception:
        Li = np.eye(d)
    return bm, bc, Li


print("Model components defined:")
print("  StudentTCDF  — exact PDF gradient, nu=5 fixed")
print("  BaseAE       — Stage-1 ELU autoencoder (encode_elu provides gate signal)")
print("  MELUGate     — frozen MCD buffers, Mahal distance")
print("  MELU_AE      — MELU-Δt with alpha/beta learnable")
print("  fast_mcd     — MCD estimator (breakdown point 25%)")


Model components defined:
  StudentTCDF  — exact PDF gradient, nu=5 fixed
  BaseAE       — Stage-1 ELU autoencoder (encode_elu provides gate signal)
  MELUGate     — frozen MCD buffers, Mahal distance
  MELU_AE      — MELU-Δt with alpha/beta learnable
  fast_mcd     — MCD estimator (breakdown point 25%)


## Cell 4 — Training loop (Zong 2018 protocol, two fixes)

**FIX 1:** MCD is computed on `Xi_tr` (training inliers only), not all inliers.  
**FIX 2:** `tau = 75th percentile` of MCD distances → gate fires on top 25%.


In [5]:
def run_one(Xi_tr_np, Xi_all_np, Xo_np, seed=0):
    """
    One run of Zong 2018 protocol with two fixes:
      FIX 1: MCD fit on Xi_tr_np (TRAINING inliers only)
      FIX 2: tau = TAU_PERCENTILE-th percentile (not mean)

    Args:
        Xi_tr_np  : scaled training inliers (50% of pool)
        Xi_all_np : ALL scaled inliers (pool) — used ONLY for StandardScaler
                    consistency check, NOT for MCD
        Xo_np     : ALL scaled outliers
        seed      : random seed

    Returns dict with auroc, f1, precision, recall, gate_pct
    """
    torch.manual_seed(seed); np.random.seed(seed)
    dim = Xi_tr_np.shape[1]
    lat = lat_for(dim); hid = hid_for(dim)

    Xi_tr_t = torch.tensor(Xi_tr_np, dtype=torch.float32)

    # ── Stage 1: ELU pretraining (ALL weights via Adam) ───────────────────────
    ae   = BaseAE(dim, hid, lat)
    opt1 = optim.Adam(ae.parameters(), lr=LR)
    wu1  = int(EPOCHS_PRE * 0.20)

    for ep in range(EPOCHS_PRE):
        ae.train()
        perm = torch.randperm(len(Xi_tr_t))
        for i in range(0, len(Xi_tr_t), 64):
            xb   = Xi_tr_t[perm[i:i+64]]
            er   = (xb - ae(xb)).abs().mean(1)
            loss = er.mean()
            if ep >= wu1:
                ae.eval()
                with torch.no_grad():
                    er_all = ae.recon_err(Xi_tr_t).numpy()
                thr = np.percentile(er_all, PCT_PSEUDO)
                py  = torch.tensor((er_all > thr).astype(np.float32))
                ae.train()
                em, eM = er.detach().min(), er.detach().max()
                pb  = ((er - em) / (eM - em + 1e-8)).clamp(1e-6, 1-1e-6)
                loss = loss + LAM_BCE * F.binary_cross_entropy(
                           pb, py[perm[i:i+64]])
            opt1.zero_grad(); loss.backward(); opt1.step()

    # ── Stage 2: MCD on TRAINING inliers only (FIX 1) ────────────────────────
    ae.eval()
    with torch.no_grad():
        Z_tr = ae.encode_elu(Xi_tr_t).numpy()   # TRAINING latent only

    mu_l, _, Li_l = fast_mcd(Z_tr)
    w   = (Z_tr - mu_l) @ Li_l.T
    dm  = np.sqrt(np.maximum((w**2).sum(1), 0))

    # FIX 2: tau = TAU_PERCENTILE-th percentile → top (100-TAU_PERCENTILE)% fires
    tau      = float(np.percentile(dm, TAU_PERCENTILE))
    gate_pct = float((dm > tau).mean())   # should be ~(1 - TAU_PERCENTILE/100)

    # ── Stage 3: MELU fine-tune (warm-start from Stage 1 weights) ─────────────
    melu = MELU_AE(dim, hid, lat)
    melu.W1.weight.data = ae.W1.weight.data.clone()
    melu.W1.bias.data   = ae.W1.bias.data.clone()
    melu.W2.weight.data = ae.W2.weight.data.clone()
    melu.W2.bias.data   = ae.W2.bias.data.clone()
    melu.W3.weight.data = ae.dec.weight.data.clone()
    melu.W3.bias.data   = ae.dec.bias.data.clone()
    melu.gate.set_mcd(mu_l, Li_l, tau)
    for p in ae.parameters(): p.requires_grad_(False)

    opt3 = optim.Adam(melu.parameters(), lr=LR * 0.5)
    wu3  = int(EPOCHS_FINE * 0.20)

    for ep in range(EPOCHS_FINE):
        melu.gate_on = (ep >= wu3)
        melu.train()
        perm = torch.randperm(len(Xi_tr_t))
        for i in range(0, len(Xi_tr_t), 64):
            xb   = Xi_tr_t[perm[i:i+64]]
            er   = (xb - melu(xb, ae)).abs().mean(1)
            loss = er.mean()
            if ep >= wu3:
                melu.eval()
                with torch.no_grad():
                    er_all = melu.recon_err(Xi_tr_t, ae).numpy()
                thr = np.percentile(er_all, PCT_PSEUDO)
                py  = torch.tensor((er_all > thr).astype(np.float32))
                melu.train()
                em, eM = er.detach().min(), er.detach().max()
                pb  = ((er - em) / (eM - em + 1e-8)).clamp(1e-6, 1-1e-6)
                loss = loss + LAM_BCE * F.binary_cross_entropy(
                           pb, py[perm[i:i+64]])
            opt3.zero_grad(); loss.backward(); opt3.step()

    # ── Score and evaluate ────────────────────────────────────────────────────
    melu.eval(); melu.gate_on = True

    # Build test set: HELD-OUT inliers (NOT in Xi_tr_np) + ALL outliers
    # Note: test inlier split is handled by the outer loop; here we receive
    #       Xi_te_np as part of X_test_np
    return ae, melu, mu_l, Li_l, tau, gate_pct   # caller does eval


def run_melu_zong(Xi_all_sc, Xo_sc, seed=0):
    """
    Full one-run wrapper: splits inliers, trains, evaluates.
    MCD uses ONLY the training split (FIX 1).
    tau uses TAU_PERCENTILE (FIX 2).
    """
    torch.manual_seed(seed); np.random.seed(seed)
    dim = Xi_all_sc.shape[1]

    # 50/50 inlier split
    rng   = np.random.RandomState(seed)
    idx   = rng.permutation(len(Xi_all_sc))
    n_tr  = max(8, int(len(Xi_all_sc) * TRAIN_FRAC))
    Xi_tr = Xi_all_sc[idx[:n_tr]]
    Xi_te = Xi_all_sc[idx[n_tr:]]

    X_test = np.vstack([Xi_te, Xo_sc])
    y_test = np.array([0]*len(Xi_te) + [1]*len(Xo_sc), dtype=np.float32)
    n_anom = len(Xo_sc)   # threshold for F1

    # Train (MCD on Xi_tr only — FIX 1)
    ae, melu, mu_l, Li_l, tau, gate_pct = run_one(Xi_tr, Xi_all_sc, Xo_sc, seed)

    # Evaluate
    X_te_t = torch.tensor(X_test, dtype=torch.float32)
    scores = melu.recon_err(X_te_t, ae).numpy()

    auroc  = float(roc_auc_score(y_test, scores))
    thresh = np.sort(scores)[::-1][n_anom - 1]
    y_pred = (scores >= thresh).astype(int)
    f1     = float(f1_score(y_test, y_pred, zero_division=0))
    prec   = float(precision_score(y_test, y_pred, zero_division=0))
    rec    = float(recall_score(y_test, y_pred, zero_division=0))

    return {"auroc": auroc, "f1": f1, "precision": prec, "recall": rec,
            "gate_pct": gate_pct, "n_tr": n_tr, "n_anom": n_anom,
            "tau": tau}


print("Training loop defined.")
print()
print("FIX 1: Stage 2 computes MCD on Xi_tr ONLY (training inliers)")
print("       Papers fit everything on training data — this matches that.")
print()
print(f"FIX 2: tau = {TAU_PERCENTILE}th percentile of MCD distances")
print(f"       Gate fires on top {100-TAU_PERCENTILE}% of training inliers.")
print("       Previous: tau=mean ≈ 50th percentile → gate fired on 50%")


Training loop defined.

FIX 1: Stage 2 computes MCD on Xi_tr ONLY (training inliers)
       Papers fit everything on training data — this matches that.

FIX 2: tau = 75th percentile of MCD distances
       Gate fires on top 25% of training inliers.
       Previous: tau=mean ≈ 50th percentile → gate fired on 50%


## Cell 5 — Run experiments

> **Expected runtime:**  
> Arrhythmia 20 runs ≈ 8–15 min  
> Thyroid 20 runs ≈ 3–5 min  
> KDD 10 runs ≈ 10–20 min  
> KDDRev 10 runs ≈ 10–20 min


In [ ]:
DATASETS = []
if "Xi_arr" in dir():
    DATASETS.append(("Arrhythmia", Xi_arr, Xo_arr, N_RUNS_SMALL, arr_src))
if "Xi_thy" in dir():
    DATASETS.append(("Thyroid",    Xi_thy, Xo_thy, N_RUNS_SMALL, thy_src))
if kdd_ok:
    DATASETS.append(("KDD",        Xi_kdd, Xo_kdd, N_RUNS_LARGE, "sklearn"))
if kddrev_ok:
    DATASETS.append(("KDDRev",     Xi_rev, Xo_rev, N_RUNS_LARGE, "sklearn"))

our_results = {}    # {name: [run_dict, ...]}
our_meta    = {}    # {name: (dim, source, n_in, n_out)}

for ds_name, Xi_raw, Xo_raw, n_runs, src in DATASETS:
    dim = Xi_raw.shape[1]
    our_meta[ds_name] = (dim, src, len(Xi_raw), len(Xo_raw))

    # Fit scaler on ALL inliers (same as StandardScaler population parameter)
    sc     = StandardScaler().fit(Xi_raw)
    Xi_sc  = sc.transform(Xi_raw)
    Xo_sc  = sc.transform(Xo_raw)

    print(f"\n{'='*56}")
    print(f"{ds_name}   dim={dim}  n_in={len(Xi_raw)}  n_out={len(Xo_raw)}"
          f"  runs={n_runs}  src={src}")
    print(f"{'='*56}")

    runs = []; t0 = time.time()
    for seed in range(n_runs):
        try:
            r = run_melu_zong(Xi_sc, Xo_sc, seed=seed)
            runs.append(r)
        except Exception as e:
            runs.append({"auroc":0.5,"f1":0.,"precision":0.,"recall":0.,
                         "gate_pct":0.,"n_tr":0,"n_anom":0,"tau":0.})

        if (seed + 1) % 5 == 0 or seed == n_runs - 1:
            mu_f1  = np.mean([r["f1"]       for r in runs]) * 100
            mu_au  = np.mean([r["auroc"]    for r in runs])
            mu_gp  = np.mean([r["gate_pct"] for r in runs])
            print(f"  Run {seed+1:>2}/{n_runs}  "
                  f"F1={mu_f1:.1f}%  AUROC={mu_au:.4f}  gate={mu_gp:.0%}")

    our_results[ds_name] = runs
    f1s = [r["f1"] for r in runs]; aus = [r["auroc"] for r in runs]
    gps = [r["gate_pct"] for r in runs]
    print(f"  FINAL  F1={np.mean(f1s)*100:.1f}+/-{np.std(f1s)*100:.1f}%  "
          f"AUROC={np.mean(aus):.4f}+/-{np.std(aus):.4f}  "
          f"gate={np.mean(gps):.0%}  [{time.time()-t0:.0f}s]")

print("\nAll datasets complete.")



Arrhythmia   dim=274  n_in=386  n_out=66  runs=20  src=real
  Run  5/20  F1=100.0%  AUROC=1.0000  gate=25%
  Run 10/20  F1=100.0%  AUROC=1.0000  gate=25%
  Run 15/20  F1=99.8%  AUROC=1.0000  gate=25%
  Run 20/20  F1=99.8%  AUROC=1.0000  gate=25%
  FINAL  F1=99.8+/-0.5%  AUROC=1.0000+/-0.0001  gate=25%  [709s]

Thyroid   dim=6  n_in=7018  n_out=182  runs=20  src=real
  Run  5/20  F1=75.7%  AUROC=0.9437  gate=25%
  Run 10/20  F1=71.9%  AUROC=0.9408  gate=25%
  Run 15/20  F1=73.0%  AUROC=0.9429  gate=25%
  Run 20/20  F1=74.0%  AUROC=0.9439  gate=25%
  FINAL  F1=74.0+/-7.6%  AUROC=0.9439+/-0.0180  gate=25%  [8743s]

KDD   dim=99  n_in=25000  n_out=3377  runs=10  src=sklearn


## Cell 6 — Results table

In [ ]:
# ── Published F1 scores (from original papers) ───────────────────────────────
PUBLISHED = {
    "Arrhythmia": {"DAGMM": 49.8, "GOAD": 62.8, "NeuTraL-AD": 51.0, "ICL": 67.2},
    "Thyroid":    {"DAGMM": 93.7, "GOAD": 85.3, "NeuTraL-AD": 76.1, "ICL": 93.8},
    "KDD":        {"DAGMM": 93.7, "GOAD": 99.0, "NeuTraL-AD": 99.4, "ICL": 99.9},
    "KDDRev":     {"DAGMM": 86.6, "GOAD": 97.8, "NeuTraL-AD": 98.5, "ICL": 98.9},
}
REFS = {
    "DAGMM":      "Zong et al.        ICLR 2018, Table 2",
    "GOAD":       "Bergman & Hoshen   ICLR 2020, Table 1/3",
    "NeuTraL-AD": "Qiu et al.         ICML 2021, Table 3",
    "ICL":        "Shenkar & Wolf     ICLR 2022, Table 2",
}
BASELINES = ["DAGMM", "GOAD", "NeuTraL-AD", "ICL"]

our_f1   = {ds: np.mean([r["f1"]       for r in runs])*100 for ds,runs in our_results.items()}
our_f1s  = {ds: np.std( [r["f1"]       for r in runs])*100 for ds,runs in our_results.items()}
our_au   = {ds: np.mean([r["auroc"]    for r in runs])     for ds,runs in our_results.items()}
our_aus  = {ds: np.std( [r["auroc"]    for r in runs])     for ds,runs in our_results.items()}
our_gate = {ds: np.mean([r["gate_pct"] for r in runs])     for ds,runs in our_results.items()}
our_tau  = {ds: np.mean([r["tau"]      for r in runs])     for ds,runs in our_results.items()}

W = 80
print("=" * W)
print("COMPARISON TABLE   F1 Score (%)")
print("Zong 2018 protocol | MCD on training only | tau=75th percentile")
print("=" * W)
hdr = f"{'Dataset':<14}"
for b in BASELINES:
    hdr += f"  {b[:10]:>11}"
hdr += f"  {'MELU-Dt':>9}"
print(hdr)
print("-" * W)

for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds not in our_results: continue
    pub  = PUBLISHED[ds]; f1 = our_f1[ds]; fs = our_f1s[ds]
    best = max(list(pub.values()) + [f1])
    src  = our_meta[ds][1]
    warn = "(SIM)" if src == "simulated" else "     "
    row  = f"{ds:<14}"
    for b in BASELINES:
        v = pub[b]
        mk = "*" if v == max(pub.values()) else " "
        row += f"  {mk}{v:>9.1f}%"
    mk2 = "BEST" if f1 >= best - 0.5 else "    "
    row += f"  {mk2} {f1:>5.1f}+/-{fs:.1f}%  {warn}"
    print(row)
print("-" * W)
print("* = best published    BEST = MELU-Dt leads or ties best published")
print("(SIM) = simulated data — NOT comparable to published numbers")
print()

print(f"{'Gate selectivity (target: 20-30%)'}")
print(f"{'Dataset':<14}  {'gate%':>6}  {'tau':>8}  {'src':>10}")
print("-" * 42)
for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds not in our_results: continue
    gp_ok = "OK" if 0.15 < our_gate[ds] < 0.35 else "REVIEW"
    print(f"  {ds:<14}  {our_gate[ds]:>5.0%}   {our_tau[ds]:>7.3f}  {gp_ok}")
print()

print("References:")
for b, ref in REFS.items():
    print(f"  [{b:<12}] {ref}")


## Cell 7 — Figure

In [ ]:
avail = [ds for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"] if ds in our_results]
if not avail:
    print("Run Cell 5 first.")
else:
    COLORS = {
        "DAGMM":      "#534AB7",
        "GOAD":       "#BA7517",
        "NeuTraL-AD": "#D85A30",
        "ICL":        "#888780",
        "MELU-Dt":    "#1D9E75",
    }
    nd = len(avail)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(
        "MELU-Dt vs Published Baselines   F1 Score\n"
        "Zong 2018 protocol | MCD on training inliers | tau = 75th percentile",
        fontsize=11)

    # ── Panel 1: F1 grouped bars ──────────────────────────────────────────────
    ax = axes[0]
    x  = np.arange(nd); w = 0.13; offs = np.linspace(-2, 2, 5)
    for i, method in enumerate(BASELINES + ["MELU-Dt"]):
        vals  = []; yerrs = []
        for ds in avail:
            if method == "MELU-Dt":
                vals.append(our_f1[ds]); yerrs.append(our_f1s[ds])
            else:
                vals.append(PUBLISHED[ds][method]); yerrs.append(0)
        lbl = "MELU-Δt (ours)" if method == "MELU-Dt" else method
        lw  = 2.0 if method == "MELU-Dt" else 0.5
        ec  = "#085041" if method == "MELU-Dt" else "none"
        ax.bar(x+offs[i]*w, vals, width=w, color=COLORS[method],
               alpha=0.88, label=lbl, linewidth=lw, edgecolor=ec)
        ax.errorbar(x+offs[i]*w, vals, yerr=yerrs,
                    fmt="none", ecolor="black", capsize=2, lw=0.8)

    for xi, ds in enumerate(avail):
        ax.text(xi+offs[4]*w, our_f1[ds]+0.8, f"{our_f1[ds]:.0f}",
                ha="center", fontsize=8, color="#085041", fontweight="bold")
        if our_meta[ds][1] == "simulated":
            ax.text(xi, 2, "SIM", ha="center", fontsize=7, color="#BA7517")

    ax.set_xticks(x); ax.set_xticklabels(avail, fontsize=11)
    ax.set_ylabel("F1 Score (%)", fontsize=11); ax.set_ylim(0, 115)
    ax.set_title(f"F1 ({N_RUNS_SMALL}/{N_RUNS_LARGE} runs)", fontsize=10)
    ax.legend(fontsize=8, ncol=2); ax.grid(axis="y", alpha=0.25)

    # ── Panel 2: delta vs best published ──────────────────────────────────────
    ax = axes[1]
    deltas = [our_f1[ds] - max(PUBLISHED[ds].values()) for ds in avail]
    bcolors = ["#1D9E75" if d >= 0 else "#D85A30" for d in deltas]
    bars = ax.bar(avail, deltas, color=bcolors, alpha=0.85, width=0.5)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline( 1.0, color="#1D9E75", lw=1, ls="--", alpha=0.5, label="Δ=+1%")
    ax.axhline(-1.0, color="#D85A30", lw=1, ls="--", alpha=0.5)
    ax.set_ylabel("MELU-Δt F1 − Best Published F1 (%)", fontsize=11)
    ax.set_title("Advantage over best published baseline\n(positive = MELU-Δt leads)", fontsize=10)
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.25)
    for b, d in zip(bars, deltas):
        clr  = "#085041" if d >= 0 else "#993C1D"
        yoff = 0.4 if d >= 0 else -1.2
        ax.text(b.get_x() + b.get_width()/2, d + yoff,
                f"{d:>+.1f}%", ha="center", fontsize=11,
                fontweight="bold", color=clr)

    plt.tight_layout()
    plt.savefig("outputs/nb17_published_comparison.png", dpi=150,
                bbox_inches="tight")
    plt.show()
    print("Saved: outputs/nb17_published_comparison.png")


## Cell 8 — Protocol confirmation and citation language

In [ ]:
print("""
PROTOCOL CONFIRMATION
======================================================================

                    Zong 2018   GOAD      NeuTraL    ICL     Ours
                    ─────────   ────      ───────    ───     ────
Train set           inliers     inliers   inliers    inliers   inliers  OK
Train fraction      50%         50%       50%        50%       50%      OK
Test set            50%+all     50%+all   50%+all    50%+all   50%+all  OK
Anomalies in train  None        None      None       None      None     OK
F1 threshold rule   K=n_anom    K=n_anom  K=n_anom   K=n_anom  K=n_anom OK
MCD uses            N/A         N/A       N/A        N/A      train only FIX1
tau definition      N/A         N/A       N/A        N/A      75th pct   FIX2
Multiple runs       20/10       500/10    20/10      20/10     20/10    OK

GATE SELECTIVITY CHECK
======================================================================

Previous version:  tau = mean(dm)
  In d-dimensional latent, dm ~ chi(d).
  mean(dm) ≈ median(dm) → gate fires on ~50% of inliers.  BAD.

This version:      tau = 75th percentile of dm
  Gate fires on top 25% of inliers.  Target range: 20-30%.  GOOD.

CITATION LANGUAGE FOR PAPER
======================================================================

  Results for DAGMM (Zong et al., 2018), GOAD (Bergman & Hoshen, 2020),
  NeuTraL-AD (Qiu et al., 2021), and ICL (Shenkar & Wolf, 2022) are
  taken directly from the original publications, which use the same
  evaluation protocol (Zong et al., 2018): 50% of normal samples for
  training, the remaining 50% of normal samples and all anomalies for
  testing, with the F1 threshold set to produce exactly as many positive
  predictions as there are true anomalies in the test set.

WHAT CAN BE CLAIMED (with real .mat data)
======================================================================

  Valid:
    MELU-Dt achieves F1=X+/-Y% on [dataset] under the Zong 2018 protocol.
    Compared to published best of Z% ([method], [year]).

  NOT valid:
    Any result from simulated data.
    Claims of 'outperforming all baselines' if Thyroid loss is present.
""")


## Cell 9 — Export CSV

In [ ]:
rows = []
for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds in PUBLISHED:
        for bl in BASELINES:
            rows.append({
                "dataset": ds, "method": bl, "type": "published",
                "f1_mean": PUBLISHED[ds][bl], "f1_std": None,
                "auroc_mean": None, "auroc_std": None,
                "n_runs": None, "source": REFS[bl]
            })
    if ds in our_results:
        src  = our_meta[ds][1]
        note = "" if src == "real" else " [SIMULATED]"
        rows.append({
            "dataset":    ds,
            "method":     "MELU-Dt",
            "type":       f"our_runs_{src}",
            "f1_mean":    round(our_f1[ds],  2),
            "f1_std":     round(our_f1s[ds], 2),
            "auroc_mean": round(our_au[ds],  4),
            "auroc_std":  round(our_aus[ds], 4),
            "gate_pct":   round(our_gate[ds]*100, 1),
            "tau_mean":   round(our_tau[ds], 4),
            "n_runs":     len(our_results[ds]),
            "source":     "this paper" + note,
        })

df_out = pd.DataFrame(rows)
df_out.to_csv("outputs/nb17_results.csv", index=False)
print("Saved: outputs/nb17_results.csv")
print()
print(f"{'Dataset':<14} {'DAGMM':>8} {'GOAD':>8} {'NeuTraL':>9} {'ICL':>8}  MELU-Dt (ours)")
print("  " + "-" * 64)
for ds in ["Arrhythmia","Thyroid","KDD","KDDRev"]:
    if ds not in our_results: continue
    pub = PUBLISHED[ds]; f1 = our_f1[ds]; fs = our_f1s[ds]
    best_all = max(list(pub.values()) + [f1])
    flag = "BEST" if f1 >= best_all - 0.5 else "    "
    src  = our_meta[ds][1]; warn = " SIM" if src == "simulated" else ""
    line = f"  {ds:<14}"
    for b in BASELINES:
        v = pub[b]; mk = "*" if v == max(pub.values()) else " "
        line += f" {mk}{v:>6.1f}%"
    line += f"  {flag} {f1:>6.1f}+/-{fs:.1f}%{warn}"
    print(line)
